# Phase 1 - Validate Kafka-Streamed Original Weather Storage

This notebook validates the streaming replay ingestion path:

`original ARCO-ERA5 MinIO partition -> Spark replay slice -> Kafka -> consumer -> MinIO JSONL micro-batches`

In [1]:
import json
import os
from io import BytesIO

import boto3
import pandas as pd
from IPython.display import display

RAW_BUCKET = os.environ["RAW_BUCKET"]
STREAM_PREFIX = "kafka_weather_events_original/"
s3 = boto3.client(
    "s3",
    endpoint_url=os.environ["MINIO_ENDPOINT_INTERNAL"],
    aws_access_key_id=os.environ["MINIO_ROOT_USER"],
    aws_secret_access_key=os.environ["MINIO_ROOT_PASSWORD"],
    region_name=os.getenv("AWS_REGION", "us-east-1"),
)

In [2]:
objects = []
paginator = s3.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket=RAW_BUCKET, Prefix=STREAM_PREFIX):
    objects.extend(page.get("Contents", []))

assert objects, "No streamed objects found. Run the consumer and producer notebooks first."
latest_run = sorted({obj["Key"].split("/")[1] for obj in objects})[-1]
latest_objects = [obj for obj in objects if f"/{latest_run}/" in obj["Key"]]
print("Latest run:", latest_run)
print("JSONL objects in latest run:", len(latest_objects))

Latest run: run_id=20260530T223925Z-92b5ef3d
JSONL objects in latest run: 5


In [3]:
rows = []
for obj in latest_objects:
    body = s3.get_object(Bucket=RAW_BUCKET, Key=obj["Key"])["Body"].read().decode("utf-8")
    rows.extend(json.loads(line) for line in body.splitlines() if line.strip())

streamed_df = pd.DataFrame(rows)
required_columns = [
    "airport_key", "timestamp_utc", "temperature_c", "wind_speed_kts",
    "precipitation_mm", "surface_pressure_pa",
]
missing = sorted(set(required_columns) - set(streamed_df.columns))
assert not missing, f"Missing streamed columns: {missing}"
display(streamed_df.head(10))

,airport_key,timestamp_utc,temperature_c,wind_speed_kts,precipitation_mm,surface_pressure_pa
0,0,2024-01-01 00:00:00,5.094,1.773,0.008386,101263.828125
1,1,2024-01-01 00:00:00,-0.444,3.473,0.000000,90422.601562
2,2,2024-01-01 00:00:00,-2.580,19.867,0.648143,99646.593750
3,3,2024-01-01 00:00:00,9.621,6.072,0.000000,99181.625000
4,4,2024-01-01 00:00:00,-4.281,8.319,0.000000,99410.718750
5,5,2024-01-01 00:00:00,4.754,8.562,0.000000,97940.273438
6,6,2024-01-01 00:00:00,9.160,3.973,0.000000,86662.914062
7,7,2024-01-01 00:00:00,9.772,2.637,0.000000,91017.390625
8,8,2024-01-01 00:00:00,13.875,1.392,0.000000,101125.523438
9,9,2024-01-01 00:00:00,11.870,4.551,0.000000,89996.656250


In [4]:
summary = {
    "streamed_rows": len(streamed_df),
    "unique_airports": int(streamed_df["airport_key"].nunique()),
    "min_timestamp_utc": streamed_df["timestamp_utc"].min(),
    "max_timestamp_utc": streamed_df["timestamp_utc"].max(),
    "jsonl_objects": len(latest_objects),
}
display(pd.DataFrame([summary]))
print("Phase 1 validation complete: original weather records flowed through Kafka into MinIO.")

,streamed_rows,unique_airports,min_timestamp_utc,max_timestamp_utc,jsonl_objects
0,240,240,2024-01-01 00:00:00,2024-01-01 00:00:00,5


Phase 1 validation complete: original weather records flowed through Kafka into MinIO.
